# Fusion : Méthode statistique + Autoencodeur

In [ ]:
import pandas as pd
import numpy as np

# 1. CHARGEMENT DES DONNÉES DES DEUX MODÈLES
df_stat = pd.read_csv('stat_annotated_timeseries.csv')
# df_ae = pd.read_csv('ae_annotated_timeseries.csv') 

# On s'assure que le temps est au format datetime
df_stat['timestamp'] = pd.to_datetime(df_stat['timestamp'])

# 2. CALCUL DES FEATURES DE FUSION 
df_story = pd.read_csv('shadow_story_fusion.csv')

def get_fusion_features(df_fusion, df_story):
    print("--- Calcul des 8 Features de Fusion ---")
    
    # Jointure avec la Shadow Story pour avoir le taux de récurrence par cluster
    df = df_fusion.merge(
        df_story[['cluster_id', 'taux_recurrence', 'intensite_moyenne_pct']], 
        on='cluster_id', 
        how='left'
    )
    
    # Feature 1: Persistance (Taux de récurrence du cluster associé)
    df['feat_persistence'] = df['taux_recurrence'].fillna(0)
    
    # Feature 2: Concordance (Exemple : Stat=1 ET AE=1)
    # Note: Remplacer 'anomaly_ae' par la vraie colonne de l'AE
    df['feat_concordance'] = np.where((df['anomaly_detected'] == 1), 1, 0) # À croiser avec AE
    
    # Feature 3: Intensité relative
    df['feat_intensity'] = df['shadow_score'].fillna(0)
    
    # Feature 4: Temporalité (Heure de la journée)
    df['feat_hour'] = df['hour_of_day'] / 23.0 # Normalisé
    
    return df

df_fusion_prepared = get_fusion_features(df_stat, df_story)

In [ ]:
def classify_shading(row):
    # Logique basée sur votre tableau de décision
    stat = row['anomaly_detected']
    # ae = row['anomaly_ae'] # Si disponible
    persistance = row['feat_persistence']
    
    if stat == 1:
        if persistance > 0.7:
            return "Ombrage Fixe Confirmé", 0.95
        elif persistance > 0.3:
            return "Ombrage Fixe Modéré", 0.85
        else:
            return "Événement Ponctuel / Nuage", 0.60
    return "Normal", 0.0

# Application de la classification
df_fusion_prepared[['final_decision', 'confidence_score']] = df_fusion_prepared.apply(
    lambda x: pd.Series(classify_shading(x)), axis=1
)

# Affichage des alertes prioritaires
print(df_fusion_prepared[df_fusion_prepared['confidence_score'] > 0.8].head())